# Calibration Under Compression v3.3 — Which Teacher Yields a Trustworthy Edge IDS Student?
### Multi-teacher distillation (FT-Transformer · XGBoost · **serialized-flow LLM**) — CICIoT2023 + Edge-IIoTset — GPU-aware, 5 seeds

**v2 claim (extends v1 findings):**
> v1 established on CICIoT2023 that (i) KD roughly doubles student miscalibration vs training from scratch,
> (ii) miscalibration is *generated by distillation under class imbalance*, not inherited — a well-calibrated teacher,
> and even a pre-calibrated teacher, still produces overconfident students — and (iii) a soft-binned calibration
> alignment loss beats post-hoc temperature scaling. **v2 asks the natural next question: does the answer depend on
> *what kind of teacher* you distill from?** We compare three fundamentally different teachers — an attention-based
> tabular transformer, a gradient-boosted tree ensemble (near-perfectly calibrated), and a **pretrained language model
> fine-tuned on quantile-serialized flows** (TabLLM-style; to our knowledge the first LLM-teacher distillation study
> for IoT intrusion detection) — distilling each into the *same* ~13k-parameter edge student.

**v3.3 addition (confirmatory causal arm):** `teacher_ft_selsharp` -> `student_kd_ft_selsharp` — the same
teacher's logits sharpened ×2 **only on samples where its max-softmax confidence exceeds 0.9**. This distortion is
sample-dependent, so unlike uniform sharpening it cannot be re-expressed as a change of global distillation
temperature. Pre-registered prediction (written before running): as a *systematic* overconfidence shift it will
transfer to the student (ECE degrades vs vanilla KD), whereas zero-mean jitter did not. Reported as a confirmatory
test regardless of outcome.

**v3.2 additions (hostile-review fixes):** (a) **label-smoothing baseline** `student_ls` — the rival explanation
("any soft target calibrates") must lose to KD for the teacher-transfer claim to stand; (b) **jittered-teacher arm**
— per-sample log-symmetric temperature jitter degrades teacher ECE while preserving average target softness,
replacing the sharpened arm (which is confounded with distillation temperature: x3 sharpening at KD_T=3 is
algebraically ~T=1 distillation) as the causal manipulation; (c) **second student size** (128-64) to test Pareto
inversion at realistic capacity; (d) **XGBoost latency** in the efficiency table; (e) stats-table hygiene
(degenerate zero-variance rows filtered) and an explicit note that temperature scaling is not top-1-order-preserving
in multiclass (so TS can legitimately change AURC).

**v3.1 additions:** (a) **sharpened-teacher ablation** — the same FT teacher's logits ×3 create an artificially
overconfident teacher *within the same protocol*; distilling from it closes the causal loop on calibration transfer;
(b) **α dose-response on all 5 seeds** (α ∈ {0.3, 0.7, 0.9}); (c) **paired-per-seed statistical tests** (paired t,
Wilcoxon, Cohen's d_z, Holm-corrected — pairs are seeds, never flows); (d) **risk–coverage / selective-prediction
analysis** and **edge latency table** computed offline from saved logits.

**v3 changes over v2:** (1) teacher gate fixed — per-dataset thresholds and best-attempt selection (v2's gate
misfired on the balanced protocol and returned lr/9-undertrained teachers); (2) **QuantileTransformer** preprocessing
(rank-gaussian) — the standard accuracy lever for tabular NNs; (3) larger FT teacher (d=96, 4 layers, 25 epochs) with a
**one-seed hyperparameter search** per dataset (checkpointed) that fixes the config for all seeds; (4) new
**class-weighted-CE scratch regime** to test whether the Web-class F1=0 pathology is fixable; (5) results in
`results_v3/` — a restore cell reuses the expensive v2 XGBoost and LLM-teacher artifacts (both are invariant to the
monotone preprocessing change), so only neural regimes retrain.

**v2 fixes over v1 (all four review-critical issues):**
1. **Teacher stability gate** — LR warmup + gradient clipping + best-epoch selection on a carved-out val split, with
   automatic retrain at lower LR if val accuracy < threshold (kills the seed-77 collapse class of failure).
2. **Capped-majority sampling** — majority classes are capped (default 60k) and minorities floored (default 2k where
   available), so Web/BruteForce become learnable and their calibration numbers stop being vacuous.
3. **`student_kd_xgb`** — distill the near-perfectly-calibrated XGBoost: the direct test of teacher-calibration transfer.
4. **α-sensitivity** — vanilla KD rerun at α=0.3 (seed 13) to show minority collapse isn't a hyperparameter artifact.

**Run plan for an 8-hour GPU session:** defaults run CICIoT2023 first, then Edge-IIoTset; everything is resume-safe,
so wherever the session ends, the next *Run All* (after restoring `results_v2/` + `logits_v2/`) continues.
Enable **Internet ON** in notebook settings (needed once to download the DistilBERT weights).

In [ ]:
# %% ======================== 1. CONFIG ========================
import os, json, math, time, glob, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
import torch.nn as nn
import torch.nn.functional as F

class CFG:
    SEEDS            = [13, 42, 77, 2024, 31337]
    # ---- per-dataset data budget ----
    N_TRAIN          = 150_000
    N_CAL            = 30_000
    N_TEST           = 40_000
    MAX_PER_CLASS    = 60_000        # cap majorities (DDoS) ...
    MIN_PER_CLASS    = 2_000         # ... floor minorities (as far as the raw data allows)
    VAL_FRAC         = 0.05          # carved from train, for teacher best-epoch + stability gate
    # ---- FT-Transformer teacher (defaults; may be overridden per-dataset by HP search) ----
    T_DIM, T_HEADS, T_LAYERS, T_EPOCHS = 96, 8, 4, 25
    T_LR             = 5e-4
    TEACHER_MIN_ACC  = {"ciciot2023": 0.70, "edgeiiot": 0.48, "_default": 0.50}   # per-dataset gate
    TEACHER_RETRIES  = 2
    HP_SEARCH        = True          # one-seed teacher HP search per dataset (checkpointed)
    HP_SEED          = 13
    HP_GRID          = [dict(d=64, layers=3, lr=1e-3), dict(d=64, layers=3, lr=5e-4),
                        dict(d=96, layers=4, lr=1e-3), dict(d=96, layers=4, lr=5e-4)]
    # Opt-in larger teacher configs — uncomment ONLY deliberately. Two warnings:
    # (1) they take effect only if you DELETE *_hp_teacher.json + all teacher_ft/student_*_ft artifacts,
    #     which retrains every FT-derived regime (~4-6 h);
    # (2) d=128/6-layer at full batch OOMs a T4 — the HP search auto-caps batch and skips OOM configs.
    # HP_GRID        += [dict(d=128, layers=6, lr=5e-4), dict(d=128, layers=4, lr=1e-3)]
    HP_BATCH         = 4096
    HP_EPOCHS        = 10            # short budget per search config; winner retrained at full T_EPOCHS
    # ---- edge student ----
    S_HIDDEN         = [64, 32]
    S_EPOCHS         = 25
    BATCH            = 4096
    LR               = 1e-3
    WARMUP_FRAC      = 0.1
    GRAD_CLIP        = 1.0
    # ---- KD ----
    KD_T             = 3.0
    KD_ALPHA         = 0.7
    ALPHA_GRID       = [(0.3, "a03"), (0.9, "a09")]   # dose-response, ALL seeds (v3.1)
    SHARPEN          = 3.0           # sharpened-teacher ablation (v3.1; confounded with KD_T — superseded by jitter)
    JITTER           = (0.5, 2.0)    # v3.2 causal arm: per-sample log-symmetric temp jitter (entropy-preserving on avg)
    SELSHARP_FACTOR  = 2.0           # v3.3 confirmatory arm: sharpen ONLY high-confidence samples...
    SELSHARP_CONF    = 0.9           # ...where teacher max-softmax > this. Sample-DEPENDENT => not expressible
                                     # as any global distillation temperature; pre-registered prediction:
                                     # transfers (student ECE degrades vs vanilla KD), unlike zero-mean jitter.
    LS_EPS           = 0.1           # v3.2 label-smoothing rival-explanation baseline
    MID_HIDDEN       = [128, 64]     # v3.2 second student size (Pareto-inversion check)
    # ---- calibration-aware KD ----
    CAKD_LAMBDA      = 0.5
    N_SOFT_BINS      = 15
    # ---- LLM teacher (TabLLM-style quantile serialization) ----
    RUN_LLM          = True
    LLM_NAME         = "distilbert-base-uncased"
    LLM_TOP_FEATS    = 24            # top features by ANOVA F -> keeps sequences short
    LLM_QBINS        = 10            # decile tokens per feature
    LLM_MAX_LEN      = 96
    LLM_EPOCHS       = 2
    LLM_LR           = 3e-5
    LLM_TRAIN_CAP    = 80_000        # cap LLM fine-tune rows (its KD logits still cover full train)
    # ---- other baselines ----
    RUN_XGB          = True
    RUN_TABPFN       = False
    TABPFN_FIT_N     = 8_000
    TABPFN_EVAL_N    = 15_000
    RUN_DATASETS     = ["ciciot2023", "edgeiiot"]
    OUT_DIR          = "/kaggle/working/results_v3"
    LOGIT_DIR        = "/kaggle/working/logits_v3"

os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.LOGIT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count() or 4)
if DEVICE == "cuda":
    CFG.BATCH = 8192
    LLM_BATCH = 128
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    LLM_BATCH = 32
    print("WARNING: no GPU detected — LLM teacher will be slow; consider CFG.RUN_LLM=False on CPU.")
print(f"torch {torch.__version__} | device={DEVICE} | batch={CFG.BATCH}")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(seed)


## 2. Data — auto-discovery + **capped-majority / floored-minority** sampling
Layout-independent discovery: every CSV is classified individually (path hints, then label vocabulary), so nested mounts like `/kaggle/input/datasets/<user>/<slug>/` work. For Edge-IIoTset the full `DNN-EdgeIIoT-dataset.csv` is preferred, files are **streamed in chunks with class-aware accumulation** (the DNN file is ordered by class — head-reads would miss attack families), and heavy payload/string columns are excluded **before parsing** (usecols) for memory safety, and the known leaky columns (`Attack_label`, ports, hosts, timestamps, payloads) are dropped explicitly. The sampling change is the v2 fix:
proportional sampling starved Web/BruteForce into unlearnability (F1 = 0.000 across all neural models in v1);
now majorities are capped at `MAX_PER_CLASS` and minorities floored at `MIN_PER_CLASS` (subject to availability),
with the remaining budget spread proportionally. Training loss stays **unweighted** — imbalance is the phenomenon
under study, not a nuisance to be reweighted away.

In [ ]:
# %% ======================== 2. DATA DISCOVERY & POOLS ========================
LABEL_CANDIDATES = ["label", "Label", " label", "class", "Class", "attack", "Attack",
                    "Attack_type", "attack_type"]          # NOTE: binary Attack_label is NOT a label candidate

# Known leaky / identifier columns for Edge-IIoTset (per the dataset authors' own preprocessing) + generic bans.
EDGEIIOT_LEAK = {"attack_label","frame.time","ip.src_host","ip.dst_host","arp.src.proto_ipv4",
                 "arp.dst.proto_ipv4","http.file_data","http.request.full_uri","http.request.uri.query",
                 "tcp.options","tcp.payload","tcp.srcport","tcp.dstport","udp.port","mqtt.msg",
                 "icmp.transmit_timestamp","icmp.unused","http.tls_port","dns.qry.name","mqtt.topic"}
GENERIC_LEAK = {"attack_label","label","class","attack"}   # never allow these as features under any casing

def to_cat(lbl: str) -> str:
    u = str(lbl).upper().replace("_", "").replace("-", "").replace(" ", "")
    if "BENIGN" in u or u in ("NORMAL", "0"):                            return "Benign"
    if "DDOS" in u:                                                      return "DDoS"
    if "MIRAI" in u:                                                     return "Mirai"
    if "DOS" in u:                                                       return "DoS"
    if any(k in u for k in ["RECON","SCAN","SWEEP","HOSTDISCOVERY","PING","FINGERPRINT"]): return "Recon"
    if any(k in u for k in ["SPOOF","MITM","ARP"]):                      return "Spoofing"
    if any(k in u for k in ["SQL","XSS","INJECTION","BACKDOOR","UPLOAD","HIJACK","WEB"]): return "Web"
    if any(k in u for k in ["RANSOM","MALWARE","TROJAN","BOTNET"]):      return "Malware"
    if any(k in u for k in ["BRUTE","DICTIONARY","PASSWORD"]):           return "BruteForce"
    return "Web"

CAT_ORDER = ["Benign","DDoS","DoS","Mirai","Recon","Spoofing","Web","BruteForce","Malware"]

def classify_family(path, vocab):
    p = path.lower()
    if any(k in p for k in ["ciciot", "cic-iot", "cic_iot", "cic iot"]): return "ciciot2023"
    if any(k in p for k in ["edge-iiot", "edgeiiot", "edge_iiot", "iiotset"]): return "edgeiiot"
    U = " ".join(vocab).upper()
    if "MIRAI" in U: return "ciciot2023"
    if any(k in U for k in ["RANSOM", "VULNERABILITY", "FINGERPRINT"]): return "edgeiiot"
    return None

def discover_inputs(root="/kaggle/input"):
    #  Layout-independent: classify every CSV individually (path hints first, then label vocabulary),
    #  so nested mounts like /kaggle/input/datasets/<user>/<slug>/ work.
    groups = {}
    for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            head = pd.read_csv(p, nrows=200)
        except Exception:
            continue
        head.columns = [c.strip() for c in head.columns]
        lab = next((l.strip() for l in LABEL_CANDIDATES if l.strip() in head.columns), None)
        if lab is None or len(head.columns) < 15:
            continue
        fam = classify_family(p, set(map(str, head[lab].unique())))
        if fam is None:
            fam = os.path.basename(os.path.dirname(p)).replace("-", "_")[:20]
        groups.setdefault(fam, []).append((p, lab))
    # Prefer the full DNN file for Edge-IIoTset when present (ML- file is a small subsample; avoid mixing schemas)
    if "edgeiiot" in groups:
        dnn = [(p, l) for p, l in groups["edgeiiot"] if "dnn-edgeiiot" in os.path.basename(p).lower()]
        if dnn: groups["edgeiiot"] = dnn
    return groups

GROUPS = discover_inputs()
assert GROUPS, "No labelled CSVs under /kaggle/input — attach CICIoT2023 and Edge-IIoTset mirrors."
print("Detected datasets:", {k: [os.path.basename(p) for p, _ in v] for k, v in GROUPS.items()})
for want in CFG.RUN_DATASETS:
    if want not in GROUPS: print(f"WARNING: '{want}' not attached — skipped this session.")

CHUNK_ROWS = 150_000

def build_pool(name):
    #  Memory-safe chunked, class-aware sampling:
    #  - usecols excludes known leaky/heavy columns BEFORE parsing (Edge-IIoT payload strings killed RAM otherwise)
    #  - numerics downcast to float32 per chunk; object columns dropped per chunk
    #  - per-class accumulation capped at 1.2x MAX_PER_CLASS (all the sampler can ever use)
    import gc
    files = GROUPS[name]
    per_class_cap = int(CFG.MAX_PER_CLASS * 1.2)
    ban = {c.lower() for c in GENERIC_LEAK}
    if name == "edgeiiot": ban |= EDGEIIOT_LEAK
    acc, filled = {}, set()

    for p, lab in files:
        try:
            hdr = [c.strip() for c in pd.read_csv(p, nrows=0).columns]
            usecols = [c for c in hdr if c == lab or c.strip().lower() not in ban]
            dropped = sorted(set(hdr) - set(usecols))
            if dropped: print(f"  [{name}] excluded before parse: {dropped}")
            for chunk in pd.read_csv(p, usecols=usecols, low_memory=False, chunksize=CHUNK_ROWS):
                chunk.columns = [c.strip() for c in chunk.columns]
                chunk = chunk.rename(columns={lab.strip(): "label"})
                chunk["cat"] = chunk["label"].map(to_cat)
                obj = [c for c in chunk.columns if chunk[c].dtype == object and c not in ("label", "cat")]
                chunk = chunk.drop(columns=obj)
                feat_c = [c for c in chunk.columns if c not in ("label", "cat")]
                chunk[feat_c] = chunk[feat_c].apply(pd.to_numeric, errors="coerce").astype(np.float32)
                for cat, g in chunk.groupby("cat"):
                    if cat in filled: continue
                    have = sum(len(x) for x in acc.get(cat, []))
                    need = per_class_cap - have
                    if need <= 0:
                        filled.add(cat); continue
                    acc.setdefault(cat, []).append(g.iloc[:need].drop(columns=["label"]))
                del chunk
        except Exception as e:
            print("  skip", p, e)

    raw = pd.concat([g for lst in acc.values() for g in lst], ignore_index=True)
    del acc; gc.collect()
    raw = raw.sample(frac=1.0, random_state=0).reset_index(drop=True)
    raw = raw.replace([np.inf, -np.inf], np.nan)
    feat = [c for c in raw.columns if c != "cat"]
    keep = [c for c in feat if raw[c].notna().mean() > 0.7 and raw[c].nunique() > 1]
    raw[keep] = raw[keep].fillna(raw[keep].median(numeric_only=True))
    classes = [c for c in CAT_ORDER if c in set(raw["cat"])]
    raw["y"] = raw["cat"].map({c: i for i, c in enumerate(classes)})
    print(f"  [{name}] pool={len(raw):,} rows | {len(keep)} features | classes={classes}")
    print(raw["cat"].value_counts().to_string())
    out = dict(name=name, df=raw[keep + ["y", "cat"]].copy(), feat=keep, classes=classes)
    del raw; gc.collect()
    return out

POOLS = {n: build_pool(n) for n in CFG.RUN_DATASETS if n in GROUPS}
assert POOLS, "None of CFG.RUN_DATASETS is attached."


In [ ]:
# %% ======================== 3. CAPPED STRATIFIED SPLITS + VAL CARVE ========================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

def capped_budget_sample(df, budget, seed):
    """Cap majorities at MAX_PER_CLASS, floor minorities at MIN_PER_CLASS (subject to availability),
    then scale the capped takes down proportionally (respecting floors) to fit the budget."""
    counts = df["y"].value_counts().to_dict()
    take  = {c: min(n, CFG.MAX_PER_CLASS) for c, n in counts.items()}
    floor = {c: min(n, CFG.MIN_PER_CLASS) for c, n in counts.items()}
    total = sum(take.values())
    if total > budget:
        for _ in range(6):   # a few proportional passes converge
            over = sum(take.values()) - budget
            if over <= 0: break
            shrinkable = {c: take[c] - floor[c] for c in take if take[c] > floor[c]}
            pool = sum(shrinkable.values())
            if pool <= 0: break
            for c, s in shrinkable.items():
                take[c] = max(floor[c], take[c] - int(math.ceil(over * s / pool)))
    out = [df[df["y"] == c].sample(n=t, random_state=seed) for c, t in take.items()]
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def make_split(pool, seed):
    df = capped_budget_sample(pool["df"], CFG.N_TRAIN + CFG.N_CAL + CFG.N_TEST, seed)
    tr, rest = train_test_split(df, test_size=(CFG.N_CAL + CFG.N_TEST) / len(df),
                                stratify=df["y"], random_state=seed)
    cal, te  = train_test_split(rest, test_size=CFG.N_TEST / len(rest),
                                stratify=rest["y"], random_state=seed)
    tr, val  = train_test_split(tr, test_size=CFG.VAL_FRAC, stratify=tr["y"], random_state=seed)
    sc = QuantileTransformer(output_distribution="normal",
                             n_quantiles=min(1000, len(tr)),
                             subsample=200_000, random_state=seed).fit(tr[pool["feat"]].values)
    X = lambda d: sc.transform(d[pool["feat"]].values).astype(np.float32)
    y = lambda d: d["y"].values.astype(np.int64)
    print(f"  split: train={len(tr):,} val={len(val):,} cal={len(cal):,} test={len(te):,} | "
          f"train class counts: {tr['cat'].value_counts().to_dict()}")
    return (X(tr), y(tr)), (X(val), y(val)), (X(cal), y(cal)), (X(te), y(te)), tr, sc


## 4. Calibration metrics (unchanged from v1 — identical code for comparability)

In [ ]:
# %% ======================== 4. METRICS ========================
from sklearn.metrics import f1_score, accuracy_score

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def ece(conf, correct, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def adaptive_ece(conf, correct, n_bins=15):
    order = np.argsort(conf); conf, correct = conf[order], correct[order]
    e = 0.0
    for idx in np.array_split(np.arange(len(conf)), n_bins):
        if len(idx): e += (len(idx)/len(conf)) * abs(correct[idx].mean() - conf[idx].mean())
    return float(e)

def classwise_ece(probs, y, classes, n_bins=15):
    per = {}
    for k, cname in enumerate(classes):
        conf_k, corr_k = probs[:, k], (y == k).astype(float)
        bins = np.linspace(0, 1, n_bins + 1); e = 0.0
        for i in range(n_bins):
            m = (conf_k > bins[i]) & (conf_k <= bins[i+1])
            if m.sum(): e += m.mean() * abs(corr_k[m].mean() - conf_k[m].mean())
        per[cname] = float(e)
    per["mean"] = float(np.mean(list(per.values())))
    return per

def reliability_bins(conf, correct, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1); rows = []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        rows.append(dict(bin_lo=float(bins[i]), bin_hi=float(bins[i+1]), n=int(m.sum()),
                         acc=float(correct[m].mean()) if m.sum() else None,
                         conf=float(conf[m].mean()) if m.sum() else None))
    return rows

def full_report(logits, y, classes):
    probs = softmax_np(logits)
    pred, conf = probs.argmax(1), probs.max(1)
    corr = (pred == y).astype(float)
    onehot = np.eye(probs.shape[1])[y]
    return dict(
        accuracy       = float(accuracy_score(y, pred)),
        macro_f1       = float(f1_score(y, pred, average="macro")),
        ece            = ece(conf, corr),
        adaptive_ece   = adaptive_ece(conf, corr),
        classwise_ece  = classwise_ece(probs, y, classes),
        brier          = float(((probs - onehot) ** 2).sum(1).mean()),
        nll            = float(-np.log(probs[np.arange(len(y)), y] + 1e-12).mean()),
        conf_on_errors = float(conf[corr == 0].mean()) if (corr == 0).any() else None,
        per_class_f1   = {c: float(f) for c, f in zip(classes,
                          f1_score(y, pred, average=None, labels=list(range(len(classes)))))},
        reliability    = reliability_bins(conf, corr),
    )


## 5. Models & losses
Teachers: **FT-Transformer** (attention over feature tokens), **XGBoost** (calibration gold standard from v1),
**DistilBERT fine-tuned on quantile-serialized flows** (each flow → a short token sequence like
`f7_q9 f12_q1 …` over the top-K most discriminative features — TabLLM-style). All three distill into the *same*
~13k-param `EdgeMLP` student, isolating the teacher's effect on student calibration.

In [ ]:
# %% ======================== 5. MODELS & LOSSES ========================
class FTTransformer(nn.Module):
    def __init__(self, n_feat, n_classes, d=CFG.T_DIM, heads=CFG.T_HEADS, layers=CFG.T_LAYERS):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_feat, d) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(n_feat, d))
        self.cls    = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=2*d,
                                         dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, n_classes))
    def forward(self, x):
        tok = x.unsqueeze(-1) * self.weight + self.bias
        tok = torch.cat([self.cls.expand(x.size(0), -1, -1), tok], dim=1)
        return self.head(self.encoder(tok)[:, 0])

class EdgeMLP(nn.Module):
    def __init__(self, n_feat, n_classes, hidden=CFG.S_HIDDEN):
        super().__init__()
        dims, layers = [n_feat] + hidden, []
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.BatchNorm1d(b), nn.ReLU(), nn.Dropout(0.1)]
        layers += [nn.Linear(dims[-1], n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def count_params(m): return sum(p.numel() for p in m.parameters())

def soft_binned_calibration_loss(logits, y, n_bins=CFG.N_SOFT_BINS):
    probs = F.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    correct = (pred == y).float()
    centers = torch.linspace(0.5 / n_bins, 1 - 0.5 / n_bins, n_bins, device=logits.device)
    width = 1.0 / n_bins
    w = torch.clamp(1 - (conf.unsqueeze(1) - centers.unsqueeze(0)).abs() / width, min=0)
    mass = w.sum(0) + 1e-8
    bin_conf = (w * conf.unsqueeze(1)).sum(0) / mass
    bin_acc  = (w * correct.unsqueeze(1)).sum(0) / mass
    return ((mass / mass.sum()) * (bin_conf - bin_acc).abs()).sum()

def kd_loss(s_logits, t_logits, y, T=CFG.KD_T, alpha=CFG.KD_ALPHA):
    soft = F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1),
                    reduction="batchmean") * (T * T)
    return alpha * soft + (1 - alpha) * F.cross_entropy(s_logits, y)

class TemperatureScaler(nn.Module):
    def __init__(self): super().__init__(); self.log_t = nn.Parameter(torch.zeros(1))
    def forward(self, logits): return logits / torch.exp(self.log_t)
    def fit(self, logits, y, iters=300):
        logits = torch.as_tensor(np.asarray(logits)); y = torch.as_tensor(np.asarray(y))
        opt = torch.optim.LBFGS([self.log_t], lr=0.05, max_iter=iters)
        def closure():
            opt.zero_grad(); loss = F.cross_entropy(self(logits), y); loss.backward(); return loss
        opt.step(closure)
        return float(torch.exp(self.log_t))


In [ ]:
# %% ======================== 6. DEVICE-AWARE ENGINE + TEACHER STABILITY GATE ========================
def _warmup_cosine(opt, total_steps):
    warm = max(1, int(CFG.WARMUP_FRAC * total_steps))
    def fn(step):
        if step < warm: return step / warm
        p = (step - warm) / max(1, total_steps - warm)
        return 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def train_model(model, Xtr, ytr, epochs, loss_fn, tag="", Xval=None, yval=None, lr=CFG.LR, batch=None):
    """Warmup+cosine, grad clipping, best-epoch selection on val (if provided)."""
    model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch or CFG.BATCH, shuffle=True,
                                     pin_memory=(DEVICE == "cuda"))
    sched = _warmup_cosine(opt, epochs * len(dl))
    best_acc, best_state = -1.0, None
    for ep in range(epochs):
        model.train(); t0, tot, nb = time.time(), 0.0, 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            loss = loss_fn(model, xb, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            opt.step(); sched.step()
            tot += float(loss); nb += 1
        msg = f"  [{tag}] epoch {ep+1}/{epochs} loss={tot/nb:.4f}"
        if Xval is not None:
            vacc = float((get_logits(model, Xval).argmax(1) == yval).mean())
            msg += f" val_acc={vacc:.4f}"
            if vacc > best_acc:
                best_acc = vacc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(msg + f" ({time.time()-t0:.0f}s)")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc

def gate_threshold(ds_name):
    t = CFG.TEACHER_MIN_ACC
    return t.get(ds_name, t.get("_default", 0.5)) if isinstance(t, dict) else t

def train_teacher_gated(n_feat, K, Xtr, ytr, Xval, yval, tag, ds_name,
                        d=None, heads=None, layers=None, lr=None, epochs=None):
    """Stability gate v3: per-dataset threshold; retrain at lower LR on failure; ALWAYS return the
    best attempt by val accuracy (v2 returned the last attempt — bug)."""
    thr = gate_threshold(ds_name)
    lr = lr or CFG.T_LR
    best = (None, -1.0)
    for attempt in range(CFG.TEACHER_RETRIES + 1):
        teacher = FTTransformer(n_feat, K, d=d or CFG.T_DIM, heads=heads or CFG.T_HEADS,
                                layers=layers or CFG.T_LAYERS)
        teacher, vacc = train_model(teacher, Xtr, ytr, epochs or CFG.T_EPOCHS,
                                    lambda m, x, y: F.cross_entropy(m(x), y),
                                    tag=f"{tag} try{attempt}", Xval=Xval, yval=yval, lr=lr)
        if vacc > best[1]:
            best = (teacher, vacc)
        if vacc >= thr:
            return teacher, vacc
        print(f"  !! teacher below gate (val_acc={vacc:.4f} < {thr}) — retrying at lr/3")
        lr /= 3
    print(f"  !! gate not reached after retries — returning BEST attempt (val_acc={best[1]:.4f}); flagged in JSON")
    return best

def hp_search_teacher(pool, ds_name):
    """One-seed, short-budget HP search for the FT teacher; winner config checkpointed and reused for all seeds."""
    ckpt = os.path.join(CFG.OUT_DIR, f"{ds_name}_hp_teacher.json")
    if os.path.exists(ckpt):
        cfg = json.load(open(ckpt)); print(f"  [{ds_name}] HP search restored: {cfg}"); return cfg
    (Xtr, ytr), (Xval, yval), _, _, _, _ = make_split(pool, CFG.HP_SEED)
    K, n_feat = len(pool["classes"]), Xtr.shape[1]
    import gc
    results = []
    hp_batch = min(CFG.BATCH, getattr(CFG, "HP_BATCH", 4096))
    for g in CFG.HP_GRID:
        set_seed(CFG.HP_SEED)
        try:
            m = FTTransformer(n_feat, K, d=g["d"], heads=max(4, g["d"] // 16), layers=g["layers"])
            m, vacc = train_model(m, Xtr, ytr, CFG.HP_EPOCHS,
                                  lambda mm, x, y: F.cross_entropy(mm(x), y),
                                  tag=f"HP {ds_name} {g}", Xval=Xval, yval=yval,
                                  lr=g["lr"], batch=hp_batch)
            results.append((vacc, g))
        except torch.cuda.OutOfMemoryError:
            print(f"  !! HP config {g} OOM on this GPU — skipped")
        finally:
            try: del m
            except NameError: pass
            gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
    assert results, "all HP configs failed — reduce HP_GRID sizes or HP_BATCH"
    vacc, best = max(results, key=lambda t: t[0])
    best = dict(best); best["heads"] = max(4, best["d"] // 16); best["val_acc"] = vacc
    json.dump(best, open(ckpt, "w"))
    print(f"  [{ds_name}] HP winner: {best}")
    return best

@torch.no_grad()
def get_logits(model, X, bs=16384):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i+bs]).to(DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def paths(ds, seed, regime):
    return (os.path.join(CFG.OUT_DIR,   f"{ds}_seed{seed}_{regime}.json"),
            os.path.join(CFG.LOGIT_DIR, f"{ds}_seed{seed}_{regime}.npz"))

def done(ds, seed, regime): return os.path.exists(paths(ds, seed, regime)[0])

def save_run(ds, classes, seed, regime, test_logits, y_test, extra=None, cal_logits=None, y_cal=None):
    rep = full_report(test_logits, y_test, classes)
    rep.update(dict(dataset=ds, seed=seed, regime=regime, classes=classes, **(extra or {})))
    jp, npz = paths(ds, seed, regime)
    kw = {"cal_logits": cal_logits, "y_cal": y_cal} if cal_logits is not None else {}
    np.savez_compressed(npz, test_logits=test_logits, y_test=y_test, **kw)
    with open(jp, "w") as f: json.dump(rep, f, indent=1)
    print(f"  saved [{ds}] {regime} seed={seed}  acc={rep['accuracy']:.4f} "
          f"macroF1={rep['macro_f1']:.4f} ECE={rep['ece']:.4f} cwECE={rep['classwise_ece']['mean']:.4f}")
    return rep


## 6b. LLM teacher — DistilBERT on quantile-serialized flows
Each flow becomes a short sentence over its top-K most discriminative features (ANOVA F on train),
each value quantized to train-set deciles: `f7_q9 f12_q1 f3_q5 …`. DistilBERT is fine-tuned for sequence
classification, then its logits on the (full) train split become KD targets exactly like the other teachers.
Novelty note: TabLLM-style serialization exists for tabular classification generally; using a fine-tuned LLM
**as a distillation teacher for IoT intrusion detection, and auditing the calibration it induces in an edge student**,
has no precedent we could find — cite TabLLM and differentiate on both counts.

In [ ]:
# %% ======================== 6b. LLM TEACHER ========================
from sklearn.feature_selection import f_classif

def load_llm(num_labels):
    """Isolated so it can be stubbed in tests / swapped for another checkpoint."""
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(CFG.LLM_NAME)
    mdl = AutoModelForSequenceClassification.from_pretrained(CFG.LLM_NAME, num_labels=num_labels)
    return tok, mdl

def fit_serializer(Xtr, ytr):
    """Pick top-K features by ANOVA F; compute decile edges on train."""
    Fv, _ = f_classif(Xtr, ytr)
    Fv = np.nan_to_num(Fv, nan=-1)
    top = np.argsort(-Fv)[:CFG.LLM_TOP_FEATS]
    edges = {int(j): np.quantile(Xtr[:, j], np.linspace(0, 1, CFG.LLM_QBINS + 1)[1:-1]) for j in top}
    return top, edges

def serialize(X, top, edges):
    cols = []
    for j in top:
        q = np.digitize(X[:, j], edges[int(j)])
        cols.append(np.char.add(f"f{j}_q", q.astype(str)))
    return [" ".join(row) for row in zip(*cols)]

def train_llm_teacher(Xtr, ytr, Xval, yval, K, tag):
    tok, mdl = load_llm(K)
    mdl.to(DEVICE)
    top, edges = fit_serializer(Xtr, ytr)
    cap = min(len(Xtr), CFG.LLM_TRAIN_CAP)
    idx = np.random.RandomState(0).choice(len(Xtr), cap, replace=False)

    def encode(X):
        texts = serialize(X, top, edges)
        enc = tok(texts, truncation=True, padding="max_length",
                  max_length=CFG.LLM_MAX_LEN, return_tensors="np")
        return enc["input_ids"].astype(np.int64), enc["attention_mask"].astype(np.int64)

    ids_tr, am_tr = encode(Xtr[idx]); y_tr = ytr[idx]
    ds = torch.utils.data.TensorDataset(torch.from_numpy(ids_tr), torch.from_numpy(am_tr),
                                        torch.from_numpy(y_tr))
    dl = torch.utils.data.DataLoader(ds, batch_size=LLM_BATCH, shuffle=True,
                                     pin_memory=(DEVICE == "cuda"))
    opt = torch.optim.AdamW(mdl.parameters(), lr=CFG.LLM_LR, weight_decay=0.01)
    sched = _warmup_cosine(opt, CFG.LLM_EPOCHS * len(dl))
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    for ep in range(CFG.LLM_EPOCHS):
        mdl.train(); t0, tot, nb = time.time(), 0.0, 0
        for ib, ab, yb in dl:
            ib, ab, yb = ib.to(DEVICE), ab.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
                out = mdl(input_ids=ib, attention_mask=ab).logits
                loss = F.cross_entropy(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(mdl.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss); nb += 1
        print(f"  [{tag}] epoch {ep+1}/{CFG.LLM_EPOCHS} loss={tot/nb:.4f} ({time.time()-t0:.0f}s)")

    @torch.no_grad()
    def llm_logits(X, bs=256):
        mdl.eval(); ids, am = encode(X); out = []
        for i in range(0, len(ids), bs):
            ib = torch.from_numpy(ids[i:i+bs]).to(DEVICE)
            ab = torch.from_numpy(am[i:i+bs]).to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
                out.append(mdl(input_ids=ib, attention_mask=ab).logits.float().cpu().numpy())
        return np.concatenate(out).astype(np.float32)

    vacc = float((llm_logits(Xval).argmax(1) == yval).mean())
    print(f"  [{tag}] val_acc={vacc:.4f}")
    return llm_logits, vacc


## 6c. Restore reusable v2 artifacts (XGBoost + LLM teacher)
Both are invariant to the v3 preprocessing change (trees and quantile-serialization are monotone-transform
invariant), and the splits are row-identical (same pool code, same seeds). Attach your v2 checkpoint dataset
(containing `results_v2/` and `logits_v2/`) as an input; this cell copies only the reusable regimes into the v3
directories so the expensive LLM fine-tunes and XGBoost fits are not repeated. Everything neural retrains.

In [ ]:
# %% ======================== 6c. RESTORE REUSABLE V2 ARTIFACTS ========================
import shutil
REUSE_PATTERNS = ["*_xgboost.json", "*_xgboost.npz", "*_teacher_llm.json", "*_teacher_llm.npz",
                  "*_llm_train.npz", "*_xgb_train.npz"]
restored = 0
# v3 checkpoints restore wholesale (everything is resume-keyed) — prevents accidental full-grid retrains
for root in glob.glob("/kaggle/input/**/results_v3", recursive=True) + \
            glob.glob("/kaggle/input/**/logits_v3", recursive=True):
    dst = CFG.OUT_DIR if root.endswith("results_v3") else CFG.LOGIT_DIR
    for f in glob.glob(os.path.join(root, "*")):
        tgt = os.path.join(dst, os.path.basename(f))
        if not os.path.exists(tgt):
            shutil.copy(f, tgt); restored += 1
for root in glob.glob("/kaggle/input/**/results_v2", recursive=True) +             glob.glob("/kaggle/input/**/logits_v2", recursive=True):
    dst = CFG.OUT_DIR if root.endswith("results_v2") else CFG.LOGIT_DIR
    for pat in REUSE_PATTERNS:
        for f in glob.glob(os.path.join(root, pat)):
            tgt = os.path.join(dst, os.path.basename(f))
            if not os.path.exists(tgt):
                shutil.copy(f, tgt); restored += 1
print(f"restored {restored} reusable v2 artifact files" if restored else
      "no v2 checkpoints found — XGBoost and LLM teachers will train fresh")


## 7. v3 experiment grid — per dataset, per seed
| # | regime | isolates |
|---|--------|----------|
| 1 | `teacher_ft` (+`_ts`) | FT-Transformer teacher (stability-gated) |
| 2 | `xgboost` | GBDT teacher/baseline (v1's calibration gold standard) |
| 3 | `teacher_llm` | DistilBERT on serialized flows — LLM teacher reference |
| 4 | `student_scratch` (+`_ts`) | compression control |
| 5 | `student_kd_ft` (+`_ts`) | vanilla KD from transformer — v1 headline, now on fixed sampling |
| 6 | `student_cakd_ft` | soft-binned calibration-aware KD (v1 winner) |
| 7 | `student_kd_ft_precal` | KD from temperature-scaled teacher |
| 8 | `student_kd_xgb` | **does a near-perfectly-calibrated teacher rescue the student?** |
| 9 | `student_kd_llm` | **does an LLM teacher's confidence geometry transfer differently?** |
| 10 | `student_kd_ft_a03` | α=0.3 sensitivity (seed 13 only) |
| 10b | `student_scratch_wce` (+`_ts`) | class-weighted CE — is the Web-class F1=0 pathology fixable? |
| 10c | `student_kd_ft_a09` | α dose-response (with a03, all seeds) |
| 10d | `teacher_ft_sharp` → `student_kd_ft_sharp` | **causal arm**: same teacher, sharpened ×3 → does miscalibration transfer? |
| 10h | `teacher_ft_selsharp` → `student_kd_ft_selsharp` | **v3.3 confirmatory arm**: sharpen ×2 only where teacher conf > 0.9 — sample-dependent, temperature-unconfoundable |
| 11 | `tabpfn` *(optional)* | tabular foundation model reference |

Teacher-type comparison (8 vs 5 vs 9) is the v2 novelty: same student, same data, three teachers with radically
different confidence geometries.

In [ ]:
# %% ======================== 7. RUN: DATASETS x SEEDS x REGIMES ========================
ALL_START = time.time()

for ds_name, pool in POOLS.items():
    classes = pool["classes"]; K = len(classes)
    print(f"\n############ DATASET {ds_name} ({K} classes) ############")
    HP = hp_search_teacher(pool, ds_name) if CFG.HP_SEARCH else {}
    for seed in CFG.SEEDS:
        print(f"\n================ {ds_name} | SEED {seed} ================")
        set_seed(seed)
        (Xtr, ytr), (Xval, yval), (Xcal, ycal), (Xte, yte), tr_df, sc = make_split(pool, seed)
        n_feat = Xtr.shape[1]

        # ---------- FT TEACHER (gated) ----------
        t_train_path = os.path.join(CFG.LOGIT_DIR, f"{ds_name}_seed{seed}_teacher_ft_train.npz")
        if not done(ds_name, seed, "teacher_ft") or not os.path.exists(t_train_path):
            teacher, vacc = train_teacher_gated(n_feat, K, Xtr, ytr, Xval, yval,
                                                tag=f"T-ft {ds_name} s{seed}", ds_name=ds_name,
                                                d=HP.get("d"), heads=HP.get("heads"),
                                                layers=HP.get("layers"), lr=HP.get("lr"))
            t_test, t_cal = get_logits(teacher, Xte), get_logits(teacher, Xcal)
            t_train_logits = get_logits(teacher, Xtr)
            np.savez_compressed(t_train_path, train_logits=t_train_logits)
            save_run(ds_name, classes, seed, "teacher_ft", t_test, yte,
                     extra={"params": count_params(teacher), "val_acc": vacc,
                            "gate_passed": bool(vacc >= gate_threshold(ds_name)), "hp": HP},
                     cal_logits=t_cal, y_cal=ycal)
            del teacher
        else:
            z = np.load(paths(ds_name, seed, "teacher_ft")[1])
            t_test, t_cal = z["test_logits"], z["cal_logits"]
            t_train_logits = np.load(t_train_path)["train_logits"]
            print("  teacher_ft: restored")

        if not done(ds_name, seed, "teacher_ft_ts"):
            T_opt = TemperatureScaler().fit(t_cal, ycal)
            save_run(ds_name, classes, seed, "teacher_ft_ts", t_test / T_opt, yte, extra={"T": T_opt})

        # ---------- XGBOOST teacher/baseline ----------
        xgb_train_path = os.path.join(CFG.LOGIT_DIR, f"{ds_name}_seed{seed}_xgb_train.npz")
        if CFG.RUN_XGB and (not done(ds_name, seed, "xgboost") or not os.path.exists(xgb_train_path)):
            from xgboost import XGBClassifier
            kw = dict(n_estimators=300, max_depth=8, learning_rate=0.2, tree_method="hist",
                      n_jobs=-1, random_state=seed, objective="multi:softprob", eval_metric="mlogloss")
            if DEVICE == "cuda": kw["device"] = "cuda"
            xgb = XGBClassifier(**kw)
            xgb.fit(Xtr, ytr)
            xgb_train_logits = np.log(np.clip(xgb.predict_proba(Xtr), 1e-9, 1.0)).astype(np.float32)
            np.savez_compressed(xgb_train_path, train_logits=xgb_train_logits)
            save_run(ds_name, classes, seed, "xgboost",
                     np.log(np.clip(xgb.predict_proba(Xte), 1e-9, 1.0)), yte)
            del xgb
        elif os.path.exists(xgb_train_path):
            xgb_train_logits = np.load(xgb_train_path)["train_logits"]
            print("  xgboost: restored")

        # ---------- LLM teacher ----------
        llm_train_path = os.path.join(CFG.LOGIT_DIR, f"{ds_name}_seed{seed}_llm_train.npz")
        if CFG.RUN_LLM and (not done(ds_name, seed, "teacher_llm") or not os.path.exists(llm_train_path)):
            llm_logits_fn, llm_vacc = train_llm_teacher(Xtr, ytr, Xval, yval, K,
                                                        tag=f"T-llm {ds_name} s{seed}")
            llm_train_logits = llm_logits_fn(Xtr)
            np.savez_compressed(llm_train_path, train_logits=llm_train_logits)
            save_run(ds_name, classes, seed, "teacher_llm", llm_logits_fn(Xte), yte,
                     extra={"val_acc": llm_vacc, "llm": CFG.LLM_NAME},
                     cal_logits=llm_logits_fn(Xcal), y_cal=ycal)
            del llm_logits_fn
            if DEVICE == "cuda": torch.cuda.empty_cache()
        elif os.path.exists(llm_train_path):
            llm_train_logits = np.load(llm_train_path)["train_logits"]
            print("  teacher_llm: restored")

        # ---------- STUDENT SCRATCH (+TS) ----------
        if not done(ds_name, seed, "student_scratch"):
            s = EdgeMLP(n_feat, K)
            print(f"  student params: {count_params(s):,}")
            s, _ = train_model(s, Xtr, ytr, CFG.S_EPOCHS,
                               lambda m, x, y: F.cross_entropy(m(x), y),
                               tag=f"S-scratch {ds_name} s{seed}")
            save_run(ds_name, classes, seed, "student_scratch", get_logits(s, Xte), yte,
                     extra={"params": count_params(s)},
                     cal_logits=get_logits(s, Xcal), y_cal=ycal)
        if not done(ds_name, seed, "student_scratch_ts"):
            z = np.load(paths(ds_name, seed, "student_scratch")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_scratch_ts",
                     z["test_logits"] / T_opt, yte, extra={"T": T_opt})

        # ---------- STUDENT SCRATCH + WEIGHTED CE (+TS) ----------
        if not done(ds_name, seed, "student_scratch_wce"):
            cw = torch.tensor(len(ytr) / (K * np.bincount(ytr, minlength=K).clip(min=1)),
                              dtype=torch.float32, device=DEVICE)
            s = EdgeMLP(n_feat, K)
            s, _ = train_model(s, Xtr, ytr, CFG.S_EPOCHS,
                               lambda m, x, y: F.cross_entropy(m(x), y, weight=cw),
                               tag=f"S-wce {ds_name} s{seed}")
            save_run(ds_name, classes, seed, "student_scratch_wce", get_logits(s, Xte), yte,
                     cal_logits=get_logits(s, Xcal), y_cal=ycal)
        if not done(ds_name, seed, "student_scratch_wce_ts"):
            z = np.load(paths(ds_name, seed, "student_scratch_wce")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_scratch_wce_ts",
                     z["test_logits"] / T_opt, yte, extra={"T": T_opt})

        # ---------- KD helper ----------
        def train_kd_student(t_logits_np, use_cal_loss=False, alpha=CFG.KD_ALPHA, tag="S-KD", hidden=None):
            s = EdgeMLP(n_feat, K, hidden=hidden or CFG.S_HIDDEN).to(DEVICE)
            tt = torch.from_numpy(t_logits_np.astype(np.float32))
            ds_t = torch.utils.data.TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr), tt)
            dl = torch.utils.data.DataLoader(ds_t, batch_size=CFG.BATCH, shuffle=True,
                                             pin_memory=(DEVICE == "cuda"))
            opt = torch.optim.AdamW(s.parameters(), lr=CFG.LR, weight_decay=1e-4)
            sched = _warmup_cosine(opt, CFG.S_EPOCHS * len(dl))
            for ep in range(CFG.S_EPOCHS):
                s.train(); t0, tot, nb = time.time(), 0.0, 0
                for xb, yb, tb in dl:
                    xb, yb, tb = xb.to(DEVICE), yb.to(DEVICE), tb.to(DEVICE)
                    opt.zero_grad()
                    out = s(xb)
                    loss = kd_loss(out, tb, yb, alpha=alpha)
                    if use_cal_loss:
                        loss = loss + CFG.CAKD_LAMBDA * soft_binned_calibration_loss(out, yb)
                    loss.backward()
                    nn.utils.clip_grad_norm_(s.parameters(), CFG.GRAD_CLIP)
                    opt.step(); sched.step()
                    tot += float(loss); nb += 1
                print(f"  [{tag} {ds_name} s{seed}] epoch {ep+1}/{CFG.S_EPOCHS} "
                      f"loss={tot/nb:.4f} ({time.time()-t0:.0f}s)")
            return s

        def run_kd_regime(regime, t_logits_np, **kd_kw):
            if not done(ds_name, seed, regime):
                s = train_kd_student(t_logits_np, tag=regime, **kd_kw)
                save_run(ds_name, classes, seed, regime, get_logits(s, Xte), yte,
                         extra={k: v for k, v in kd_kw.items() if k not in ("use_cal_loss",)},
                         cal_logits=get_logits(s, Xcal), y_cal=ycal)

        # ---------- 5-7) KD from FT teacher ----------
        run_kd_regime("student_kd_ft", t_train_logits)
        if not done(ds_name, seed, "student_kd_ft_ts"):
            z = np.load(paths(ds_name, seed, "student_kd_ft")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_kd_ft_ts",
                     z["test_logits"] / T_opt, yte, extra={"T": T_opt})
        run_kd_regime("student_cakd_ft", t_train_logits, use_cal_loss=True)
        if not done(ds_name, seed, "student_kd_ft_precal"):
            T_teacher = TemperatureScaler().fit(t_cal, ycal)
            s = train_kd_student(t_train_logits / T_teacher, tag="student_kd_ft_precal")
            save_run(ds_name, classes, seed, "student_kd_ft_precal", get_logits(s, Xte), yte,
                     extra={"teacher_T": T_teacher},
                     cal_logits=get_logits(s, Xcal), y_cal=ycal)

        # ---------- 8) KD from XGBoost teacher ----------
        if CFG.RUN_XGB:
            run_kd_regime("student_kd_xgb", xgb_train_logits)

        # ---------- 9) KD from LLM teacher ----------
        if CFG.RUN_LLM and os.path.exists(llm_train_path):
            run_kd_regime("student_kd_llm", llm_train_logits)

        # ---------- 10) alpha dose-response (ALL seeds, v3.1) ----------
        for aval, aname in CFG.ALPHA_GRID:
            run_kd_regime(f"student_kd_ft_{aname}", t_train_logits, alpha=aval)

        # ---------- 10c) sharpened-teacher ablation (v3.1) ----------
        if not done(ds_name, seed, "teacher_ft_sharp"):
            save_run(ds_name, classes, seed, "teacher_ft_sharp",
                     t_test * CFG.SHARPEN, yte, extra={"sharpen": CFG.SHARPEN},
                     cal_logits=t_cal * CFG.SHARPEN, y_cal=ycal)
        run_kd_regime("student_kd_ft_sharp", t_train_logits * CFG.SHARPEN)

        # ---------- 10e) LABEL-SMOOTHING baseline (v3.2: rival explanation) ----------
        if not done(ds_name, seed, "student_ls"):
            s = EdgeMLP(n_feat, K)
            s, _ = train_model(s, Xtr, ytr, CFG.S_EPOCHS,
                               lambda m, x, y: F.cross_entropy(m(x), y, label_smoothing=CFG.LS_EPS),
                               tag=f"S-ls {ds_name} s{seed}")
            save_run(ds_name, classes, seed, "student_ls", get_logits(s, Xte), yte,
                     extra={"ls_eps": CFG.LS_EPS}, cal_logits=get_logits(s, Xcal), y_cal=ycal)
        if not done(ds_name, seed, "student_ls_ts"):
            z = np.load(paths(ds_name, seed, "student_ls")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_ls_ts", z["test_logits"] / T_opt, yte, extra={"T": T_opt})

        # ---------- 10f) JITTERED-TEACHER causal arm (v3.2) ----------
        rng_j = np.random.RandomState(seed)
        f_tr  = rng_j.choice(CFG.JITTER, size=len(t_train_logits)).astype(np.float32)[:, None]
        f_te  = rng_j.choice(CFG.JITTER, size=len(t_test)).astype(np.float32)[:, None]
        f_cal = rng_j.choice(CFG.JITTER, size=len(t_cal)).astype(np.float32)[:, None]
        if not done(ds_name, seed, "teacher_ft_jitter"):
            save_run(ds_name, classes, seed, "teacher_ft_jitter", t_test * f_te, yte,
                     extra={"jitter": list(CFG.JITTER)}, cal_logits=t_cal * f_cal, y_cal=ycal)
        run_kd_regime("student_kd_ft_jitter", t_train_logits * f_tr)

        # ---------- 10h) SELECTIVE-SHARPENING confirmatory arm (v3.3) ----------
        def selsharp(logits):
            conf = softmax_np(logits).max(1)
            f = np.where(conf > CFG.SELSHARP_CONF, CFG.SELSHARP_FACTOR, 1.0).astype(np.float32)[:, None]
            return logits * f, float((f > 1).mean())
        if not done(ds_name, seed, "teacher_ft_selsharp"):
            te_s, frac_te = selsharp(t_test)
            cal_s, _ = selsharp(t_cal)
            save_run(ds_name, classes, seed, "teacher_ft_selsharp", te_s, yte,
                     extra={"factor": CFG.SELSHARP_FACTOR, "conf_thresh": CFG.SELSHARP_CONF,
                            "frac_sharpened_test": frac_te},
                     cal_logits=cal_s, y_cal=ycal)
        if not done(ds_name, seed, "student_kd_ft_selsharp"):
            tr_s, frac_tr = selsharp(t_train_logits)
            s = train_kd_student(tr_s, tag="student_kd_ft_selsharp")
            save_run(ds_name, classes, seed, "student_kd_ft_selsharp", get_logits(s, Xte), yte,
                     extra={"frac_sharpened_train": frac_tr},
                     cal_logits=get_logits(s, Xcal), y_cal=ycal)

        # ---------- 10g) MID-SIZE student (v3.2: Pareto-inversion check) ----------
        if not done(ds_name, seed, "student_mid_scratch"):
            s = EdgeMLP(n_feat, K, hidden=CFG.MID_HIDDEN)
            print(f"  mid student params: {count_params(s):,}")
            s, _ = train_model(s, Xtr, ytr, CFG.S_EPOCHS,
                               lambda m, x, y: F.cross_entropy(m(x), y),
                               tag=f"S-mid {ds_name} s{seed}")
            save_run(ds_name, classes, seed, "student_mid_scratch", get_logits(s, Xte), yte,
                     extra={"params": count_params(s)}, cal_logits=get_logits(s, Xcal), y_cal=ycal)
        if not done(ds_name, seed, "student_mid_scratch_ts"):
            z = np.load(paths(ds_name, seed, "student_mid_scratch")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_mid_scratch_ts",
                     z["test_logits"] / T_opt, yte, extra={"T": T_opt})
        run_kd_regime("student_mid_kd_ft", t_train_logits, hidden=CFG.MID_HIDDEN)
        if not done(ds_name, seed, "student_mid_kd_ft_ts"):
            z = np.load(paths(ds_name, seed, "student_mid_kd_ft")[1])
            T_opt = TemperatureScaler().fit(z["cal_logits"], z["y_cal"])
            save_run(ds_name, classes, seed, "student_mid_kd_ft_ts",
                     z["test_logits"] / T_opt, yte, extra={"T": T_opt})

        # ---------- 11) TabPFN (optional) ----------
        if CFG.RUN_TABPFN and not done(ds_name, seed, "tabpfn"):
            try:
                from tabpfn import TabPFNClassifier
                fi = np.random.RandomState(seed).choice(len(Xtr), CFG.TABPFN_FIT_N, replace=False)
                ei = np.random.RandomState(seed).choice(len(Xte), CFG.TABPFN_EVAL_N, replace=False)
                clf = TabPFNClassifier(device=DEVICE); clf.fit(Xtr[fi], ytr[fi])
                p = np.clip(clf.predict_proba(Xte[ei]), 1e-9, 1.0)
                save_run(ds_name, classes, seed, "tabpfn", np.log(p), yte[ei],
                         extra={"fit_n": CFG.TABPFN_FIT_N, "eval_n": CFG.TABPFN_EVAL_N})
            except Exception as e:
                print("  TabPFN skipped:", e)

print(f"\nALL DONE in {(time.time()-ALL_START)/60:.1f} min")


## 8. Aggregate → paper-ready tables, incl. the **teacher-type comparison**

In [ ]:
# %% ======================== 8. AGGREGATE ========================
rows = []
for jp in sorted(glob.glob(os.path.join(CFG.OUT_DIR, "*_seed*_*.json"))):
    if jp.endswith("_hp_teacher.json"): continue
    with open(jp) as f: r = json.load(f)
    rows.append(dict(dataset=r["dataset"], seed=r["seed"], regime=r["regime"],
                     accuracy=r["accuracy"], macro_f1=r["macro_f1"], ece=r["ece"],
                     adaptive_ece=r["adaptive_ece"], classwise_ece=r["classwise_ece"]["mean"],
                     brier=r["brier"], nll=r["nll"], conf_on_errors=r.get("conf_on_errors"),
                     gate_passed=r.get("gate_passed"),
                     **{f"cwECE_{k}": v for k, v in r["classwise_ece"].items() if k != "mean"},
                     **{f"F1_{k}": v for k, v in r["per_class_f1"].items()}))
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/all_runs_raw_v3.csv", index=False)

main_cols = ["accuracy","macro_f1","ece","adaptive_ece","classwise_ece","brier","nll","conf_on_errors"]
ORDER = ["teacher_ft","teacher_ft_ts","teacher_llm","xgboost","student_scratch","student_scratch_ts",
         "student_kd_ft","student_kd_ft_ts","student_cakd_ft","student_kd_ft_precal",
         "student_kd_xgb","student_kd_llm","student_kd_ft_a03","student_kd_ft_a09",
         "teacher_ft_sharp","student_kd_ft_sharp","teacher_ft_jitter","student_kd_ft_jitter",
         "teacher_ft_selsharp","student_kd_ft_selsharp",
         "student_ls","student_ls_ts","student_mid_scratch","student_mid_scratch_ts",
         "student_mid_kd_ft","student_mid_kd_ft_ts",
         "student_scratch_wce","student_scratch_wce_ts","tabpfn"]

gate = res[res["regime"] == "teacher_ft"][["dataset","seed","gate_passed"]]
if (gate["gate_passed"] == False).any():
    print("!! Teacher gate failures (inspect before trusting KD rows):")
    display(gate[gate["gate_passed"] == False])

for ds_name in res["dataset"].unique():
    sub = res[res["dataset"] == ds_name]
    agg = sub.groupby("regime")[main_cols].agg(["mean","std"]).round(4)
    agg.to_csv(f"/kaggle/working/summary_v3_{ds_name}.csv")
    order = [r for r in ORDER if r in agg.index]
    print(f"\n========== {ds_name}: mean ± std over {sub['seed'].nunique()} seeds ==========")
    display(agg.loc[order])

    tcomp = [r for r in ["student_kd_ft","student_kd_xgb","student_kd_llm","student_scratch"] if r in agg.index]
    print(f"---- {ds_name}: TEACHER-TYPE COMPARISON (same student, different teacher) ----")
    display(sub[sub.regime.isin(tcomp)].groupby("regime")[
        ["accuracy","macro_f1","ece","classwise_ece","conf_on_errors"]].mean().round(4).loc[tcomp])

    minority = [c for c in sub.columns if c.startswith("F1_") and
                any(m in c for m in ["Recon","Spoofing","Web","BruteForce","Malware"])]
    minority = [c for c in minority if sub[c].notna().any()]
    if minority:
        print(f"---- {ds_name}: minority-class F1 (v2 sampling fix — zeros should be gone) ----")
        display(sub.groupby("regime")[minority].mean().round(3).loc[order])

print("\n========== REPLICATION VIEW ==========")
piv = res.pivot_table(index="regime", columns="dataset",
                      values=["ece","classwise_ece","macro_f1"], aggfunc="mean").round(4)
display(piv.loc[[r for r in ORDER if r in piv.index]])
piv.to_csv("/kaggle/working/replication_view_v3.csv")


## 8b. Paired-per-seed statistical tests (v3.1)
Pairs are **seeds**, never flows — 5 paired observations per comparison, per dataset. We report paired *t*,
Wilcoxon signed-rank (exact, n=5 ⇒ minimum two-sided p = 0.0625 — stated openly), Cohen's d_z, and Holm
correction within each dataset's comparison family. Interpretation leans on effect sizes + cross-dataset
replication, not stars alone.

In [ ]:
# %% ======================== 8b. STATISTICAL TESTS ========================
from scipy import stats as sps

COMPARISONS = [
    ("student_kd_ft", "student_scratch",   "ece",           "KD transfers calibration (ECE)"),
    ("student_kd_ft", "student_scratch",   "classwise_ece", "KD transfers calibration (cwECE)"),
    ("student_kd_ft_sharp", "student_kd_ft", "ece",         "sharpened teacher -> worse student ECE (causal)"),
    ("student_kd_ft", "student_scratch",   "macro_f1",      "KD minority tax (macro-F1)"),
    ("student_kd_ft_a09", "student_kd_ft_a03", "ece",       "alpha dose-response (ECE)"),
    ("student_scratch_wce", "student_scratch", "macro_f1",  "WCE minority gain"),
    ("student_scratch_wce", "student_scratch", "classwise_ece", "WCE calibration cost"),
    ("student_scratch_ts", "student_kd_ft", "ece",          "scratch+TS vs KD (ECE)"),
    ("student_ls", "student_kd_ft", "ece",                   "LS rival explanation (must NOT match KD)"),
    ("student_ls", "student_scratch", "ece",                 "LS vs scratch (soft-target effect size)"),
    ("student_kd_ft_jitter", "student_kd_ft", "ece",         "jittered teacher -> worse student ECE (causal v3.2)"),
    ("student_kd_ft_selsharp", "student_kd_ft", "ece",       "selective sharpening transfers (confirmatory v3.3)"),
    ("student_kd_ft_selsharp", "student_kd_ft", "conf_on_errors", "selective sharpening -> overconfident errors"),
    ("student_mid_kd_ft", "student_mid_scratch", "ece",      "transfer replicates at 128-64 student"),
    ("student_mid_kd_ft", "student_mid_scratch", "macro_f1", "minority tax at 128-64 student"),
]
# dataset-specific minority F1 comparisons
MINORITY = {"edgeiiot": ["F1_Spoofing", "F1_Malware"], "ciciot2023": ["F1_Web", "F1_BruteForce"]}

rows = []
for ds in res["dataset"].unique():
    comps = list(COMPARISONS) + [("student_kd_ft", "student_scratch", m, f"KD minority tax ({m})")
                                 for m in MINORITY.get(ds, []) if m in res.columns]
    pvals = []
    for regA, regB, metric, label in comps:
        A = res[(res.dataset == ds) & (res.regime == regA)].set_index("seed")[metric]
        B = res[(res.dataset == ds) & (res.regime == regB)].set_index("seed")[metric]
        common = A.index.intersection(B.index)
        if len(common) < 2: continue
        A, B = A.loc[common].astype(float), B.loc[common].astype(float)
        d = A - B
        if float(d.std(ddof=1)) == 0.0:
            print(f"  [stats] skipped degenerate zero-variance comparison: {ds} / {label}")
            continue
        t, p_t = sps.ttest_rel(A, B)
        try:
            _, p_w = sps.wilcoxon(A, B)
        except Exception:
            p_w = np.nan
        dz = d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else np.inf
        rows.append(dict(dataset=ds, comparison=label, A=regA, B=regB, metric=metric,
                         n_pairs=len(common), mean_diff=round(d.mean(), 4),
                         sd_diff=round(d.std(ddof=1), 4), cohens_dz=round(dz, 2),
                         p_ttest=p_t, p_wilcoxon=p_w))
        pvals.append(p_t)
    # Holm within this dataset family
    order = np.argsort(pvals); m = len(pvals)
    holm = {}
    running = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (m - rank) * pvals[idx]); running = max(running, adj); holm[idx] = running
    ds_rows = [r for r in rows if r["dataset"] == ds]
    for k, r in enumerate(ds_rows): r["p_holm"] = round(holm[k], 4)

stats_df = pd.DataFrame(rows)
stats_df["p_ttest"] = stats_df["p_ttest"].round(4)
stats_df["p_wilcoxon"] = stats_df["p_wilcoxon"].round(4)
stats_df.to_csv("/kaggle/working/stats_tests_v3.csv", index=False)
display(stats_df)


## 8c. Risk–coverage (selective prediction) + edge latency (v3.1)
Selective prediction is the operational payoff of calibration: an edge IDS that abstains on its least-confident
flows should hand fewer, better alerts to the SOC. AURC (area under the risk–coverage curve, lower = better) is
computed offline from saved test logits.

In [ ]:
# %% ======================== 8c. RISK-COVERAGE + LATENCY ========================
def risk_coverage(logits, y, points=50):
    probs = softmax_np(logits); conf = probs.max(1); pred = probs.argmax(1)
    order = np.argsort(-conf); err = (pred != y).astype(float)[order]
    cum = np.cumsum(err) / np.arange(1, len(err) + 1)
    idx = (np.linspace(0.02, 1.0, points) * (len(err) - 1)).astype(int)
    cov = np.linspace(0.02, 1.0, points)
    trap = getattr(np, "trapezoid", getattr(np, "trapz", None))
    return cov, cum[idx], float(trap(cum[idx], cov))

RC_REGIMES = ["student_scratch", "student_scratch_ts", "student_kd_ft", "student_kd_ft_sharp",
              "student_kd_ft_jitter", "student_kd_ft_selsharp", "student_ls", "student_mid_kd_ft", "student_mid_scratch",
              "student_kd_xgb", "student_kd_llm", "student_scratch_wce"]
print("NOTE: temperature scaling is NOT top-1-order-preserving in multiclass softmax, so *_ts AURC can "
      "legitimately differ from its base regime — state this in the paper to pre-empt a 'bug' review comment.")
aurc_rows, curves = [], {}
for ds in res["dataset"].unique():
    for reg in RC_REGIMES:
        cs = []
        for f in glob.glob(os.path.join(CFG.LOGIT_DIR, f"{ds}_seed*_{reg}.npz")):
            z = np.load(f)
            cov, risk, aurc = risk_coverage(z["test_logits"], z["y_test"])
            cs.append(risk); aurc_rows.append(dict(dataset=ds, regime=reg, aurc=aurc))
        if cs: curves[(ds, reg)] = (cov, np.mean(cs, axis=0))
aurc = pd.DataFrame(aurc_rows).groupby(["dataset", "regime"])["aurc"].agg(["mean", "std"]).round(4)
aurc.to_csv("/kaggle/working/aurc_v3.csv"); display(aurc)

import matplotlib.pyplot as plt
for ds in res["dataset"].unique():
    plt.figure(figsize=(5.5, 4))
    for reg in RC_REGIMES:
        if (ds, reg) in curves:
            cov, r = curves[(ds, reg)]; plt.plot(cov, r, label=reg, lw=1.6)
    plt.xlabel("coverage"); plt.ylabel("selective risk"); plt.title(f"risk-coverage: {ds}")
    plt.legend(fontsize=7); plt.tight_layout()
    plt.savefig(f"/kaggle/working/risk_coverage_{ds}.png", dpi=200); plt.show()

# ---- edge latency / footprint (CPU, deployment-facing) ----
lat_rows = []
for name, build in [("EdgeMLP student", lambda nf, k: EdgeMLP(nf, k)),
                    ("FT-Transformer teacher", lambda nf, k: FTTransformer(nf, k))]:
    pool0 = list(POOLS.values())[0]
    nf, k = len(pool0["feat"]), len(pool0["classes"])
    m = build(nf, k).cpu().eval()
    x1, xb = torch.randn(1, nf), torch.randn(256, nf)
    with torch.no_grad():
        for _ in range(10): m(xb)
        t0 = time.time()
        for _ in range(200): m(x1)
        lat1 = (time.time() - t0) / 200 * 1e3
        t0 = time.time()
        for _ in range(50): m(xb)
        latb = (time.time() - t0) / 50 / 256 * 1e6
    lat_rows.append(dict(model=name, params=count_params(m),
                         ms_per_flow_bs1=round(lat1, 3), us_per_flow_bs256=round(latb, 2)))
# XGBoost latency (fits a fresh model on a 100k subsample of the first pool, seed 13 split)
try:
    from xgboost import XGBClassifier
    pool0 = list(POOLS.values())[0]
    (Xtr0, ytr0), _, _, (Xte0, yte0), _, _ = make_split(pool0, CFG.SEEDS[0])
    sub = np.random.RandomState(0).choice(len(Xtr0), min(100_000, len(Xtr0)), replace=False)
    xgb_l = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.2, tree_method="hist",
                          n_jobs=-1, random_state=0, objective="multi:softprob")
    xgb_l.fit(Xtr0[sub], ytr0[sub])
    x1, xb = Xte0[:1], Xte0[:4096]
    for _ in range(3): xgb_l.predict_proba(xb)
    t0 = time.time()
    for _ in range(100): xgb_l.predict_proba(x1)
    l1 = (time.time() - t0) / 100 * 1e3
    t0 = time.time()
    for _ in range(20): xgb_l.predict_proba(xb)
    lb = (time.time() - t0) / 20 / len(xb) * 1e6
    booster = xgb_l.get_booster()
    n_nodes = sum(int(t.count("leaf")) * 2 for t in booster.get_dump())
    lat_rows.append(dict(model="XGBoost (300x8, CPU)", params=n_nodes,
                         ms_per_flow_bs1=round(l1, 3), us_per_flow_bs256=round(lb, 2)))
    del xgb_l
except Exception as e:
    print("XGB latency skipped:", e)
lat = pd.DataFrame(lat_rows); lat.to_csv("/kaggle/working/latency_v3.csv", index=False); display(lat)


In [ ]:
# %% ======================== 9. RELIABILITY DIAGRAMS (per dataset) ========================
import matplotlib.pyplot as plt

KEY = ["teacher_ft","teacher_llm","student_scratch","student_kd_ft",
       "student_kd_xgb","student_kd_llm","student_cakd_ft"]
for ds_name in res["dataset"].unique():
    sub = res[res["dataset"] == ds_name]
    key = [r for r in KEY if (sub["regime"] == r).any()]
    fig, axes = plt.subplots(1, len(key), figsize=(3.0*len(key), 3.3), sharey=True)
    for ax, reg in zip(np.atleast_1d(axes), key):
        accs, confs = [], []
        for jp in glob.glob(os.path.join(CFG.OUT_DIR, f"{ds_name}_seed*_{reg}.json")):
            with open(jp) as f: r = json.load(f)
            b = pd.DataFrame(r["reliability"]).dropna()
            accs.append(b.set_index("bin_lo")["acc"]); confs.append(b.set_index("bin_lo")["conf"])
        A = pd.concat(accs, axis=1).mean(1); Cf = pd.concat(confs, axis=1).mean(1)
        ax.plot([0,1],[0,1],"k--",lw=1)
        ax.bar(Cf.values, A.values, width=0.055, alpha=0.7, edgecolor="k")
        ax.set_title(f"{reg}\nECE={sub[sub['regime']==reg]['ece'].mean():.3f}", fontsize=8)
        ax.set_xlabel("confidence")
    np.atleast_1d(axes)[0].set_ylabel("accuracy")
    fig.suptitle(ds_name)
    plt.tight_layout(); plt.savefig(f"/kaggle/working/reliability_v3_{ds_name}.png", dpi=200)
    plt.show()
print("Saved: summary_v3_*.csv, replication_view_v3.csv, reliability_v3_*.png,")
print("       results_v3/*.json + logits_v3/*.npz (recompute ANY metric offline — no retraining).")


## 10. v3.1 session plan, freeze criteria & what remains
**Runtime on top of existing v3 checkpoints:** ~1.5–2 h (sharpened students + a03/a09 across seeds; a03 seed-13
already done and skipped). Attach your saved `results_v3/`+`logits_v3/` and copy them back before Run All.

**v3.2 runtime on top of v3.1 checkpoints:** ~1.5–2.5 h (LS, jitter, and four mid-student regimes; everything
else resumes). Framing rules locked by the hostile review: the sharpened arm is reported only as a
temperature-confounded ablation; the jittered arm carries the causal claim; if `student_ls` ≈ `student_kd_ft` on
ECE, the headline must be rewritten around soft-target calibration (decide from stats_tests, not hope); the
recommended recipe is the best α by the numbers (α=0.9 on current evidence), not the historical default; published
99%-accuracy SOTA is cited in a protocol-labeled comparison (proportional sampling / leaky features vs balanced
leakage-free) — never as a same-protocol claim.

**v3.3 runtime on top of v3.2 checkpoints:** ~40–50 min (ten selsharp students; the selsharp teacher rows are
logit arithmetic). The pre-registered prediction is recorded in the config cell; report the outcome either way.

**After this run, freeze.** Remaining for the manuscript (no GPU needed): (1) Pareto figure —
accuracy vs cwECE vs minority-F1 per student regime, both datasets; (2) reliability + risk–coverage figures
(generated here); (3) latency/params table (generated here); (4) stats table (generated here) quoted with effect
sizes, n=5 caveat stated; (5) related-work positioning: TabLLM, CD-CI, FIRCE/CALIBURN, accuracy-only KD-IDS line,
NIDS-calibration paper; (6) limitations: balanced-subsample protocol (justify vs inflated majority accuracy),
Web-class hardness for compact nets, single student architecture, DistilBERT-only LLM.
**Suggested order (defaults already do this):** CICIoT2023 first (full grid ≈ 2.5–3.5 h on a T4; LLM teacher is the
bottleneck at ~15–25 min/seed), then Edge-IIoTset. If time runs short, the resume logic means partial Edge-IIoTset
seeds are not wasted. To guarantee both datasets fit: drop `LLM_TRAIN_CAP` to 50k or `LLM_EPOCHS` to 1.

**End-of-session ritual (do not skip):** download `results_v3/` + `logits_v3/`, upload as a private Kaggle dataset;
next session copy them back into `/kaggle/working/` before *Run All*.

**Reading the teacher-type comparison (the v2 headline):**
- `student_kd_xgb` still miscalibrated ≈ `student_kd_ft` → *calibration is destroyed by distillation itself,
  regardless of teacher quality* — strongest possible version of the v1 finding.
- `student_kd_xgb` well-calibrated → *teacher confidence geometry does transfer; pick your teacher for calibration* —
  a constructive, deployable result instead.
- `student_kd_llm` behavior either way is unprecedented data (first LLM-teacher KD for IoT IDS); even parity is
  reportable as "serialized-flow LLMs are viable IDS teachers at N-times the training cost."
- `student_kd_ft_a03` ≈ `student_kd_ft` on minority-class F1 → collapse is not a hyperparameter artifact (cite in rebuttal).

**Paper positioning reminders:** cite TabLLM (serialization idea), CD-CI (calibration distillation, vision/NLP),
FIRCE + CALIBURN (conformal IDS, uncompressed), the KD-IDS accuracy-only line (Kronecker/TON_IoT, UAV, FTKD),
and the NIDS calibration + feature-selection paper — differentiate on: real-device IoT traffic, compression,
per-class calibration under imbalance, multi-teacher design.